In [2]:
import torch

## Midpoints

The scence comes with a mid-point representation. Each cell corresponds to the y, x coordinates (in Polar Stereographic) of the midpoint of the grid cell.

In [10]:
hr_scene_vel_tensor = torch.load("./torch_data/scene_vel_tensor.pt")

# vel has 2 pixel channels (index 0 and 1)
# Third channel (index 2) are y coords
hr_scene_vel_tensor[2, :, :]

# Forth channel (index 3) are x coords
hr_scene_vel_tensor[3, :, :]

tensor([[-332650., -332200., -331750.,  ..., -311500., -311050., -310600.],
        [-332650., -332200., -331750.,  ..., -311500., -311050., -310600.],
        [-332650., -332200., -331750.,  ..., -311500., -311050., -310600.],
        ...,
        [-332650., -332200., -331750.,  ..., -311500., -311050., -310600.],
        [-332650., -332200., -331750.,  ..., -311500., -311050., -310600.],
        [-332650., -332200., -331750.,  ..., -311500., -311050., -310600.]],
       dtype=torch.float64)

## Box-channels

**Box-channels** are the representation for grid box cells we will use to calculate covariances for non-subset lower-resolution grids expressed through high-resolution base covariance grids.

Box-channels have dimensitionality [4, H, W] where for now H = W. 
The four channels have the fixed order
- y_min
- y_max
- x_min
- x_max

Next we convert from a cell **midpoint representation to a box representation**.

In [131]:
def midpoint_to_box(yx_tensor):
    """_summary_

    Args:
        yx_tensor (torch.tensor): [C, H, W] tensor where C_1 is y and C_2 is x
    
    Returns:
        torch.tensor: 4-channel tensor (box-channel) y_min, y_max, x_min, x_max
    """
    # Separate both tensors
    y_tensor = yx_tensor[0, :, :] # [H, W]
    x_tensor = yx_tensor[1, :, :] # [H, W]

    # integer; y direction is reversed to obtain positive step value.
    y_step = int(y_tensor[0, 0] - y_tensor[1, 0])
    x_step = int(x_tensor[0, 1] - x_tensor[0, 0])

    ### Y MIN MAX ###
    # y_min is the lower edge of the cell.
    y_min = y_tensor - (y_step / 2)
    # y_max is the top edge of the cell.
    y_max = y_tensor + (y_step / 2)

    ### X MIN MAX ###
    # x_min is the left edge of the cell.
    x_min = x_tensor - (x_step / 2)
    # x_max is the right edge of the cell.
    x_max = x_tensor + (x_step / 2)

    return(torch.cat((y_min.unsqueeze(0),
                     y_max.unsqueeze(0),
                     x_min.unsqueeze(0), 
                     x_max.unsqueeze(0)),
                     dim = 0))

In [135]:
hr_scene_vel_box_tensor = midpoint_to_box(hr_scene_vel_tensor[2:4, :, :]) # index 2 and 3

### Upscale pixel values AND adjust box values

In [56]:
scence_vel_pixel_box_tensor = torch.cat((hr_scene_vel_tensor[:2, :, :], hr_scene_vel_box_tensor),
                                        dim = 0)

In [109]:
def upscale_box_tensor(tensor, upscaling_factor):
    """ Upscaling is the opposite of downscaling: We are increasing the scale of each grid cell represented by the value by mean aggregation. 
        From in input higher resolution tensor a lower resolution tensor is returned. 

    Args:
        tensor (torch.tensor): high-res. input 3D tensor where [C, H, W] where C is pixel_dim + box_dim (4)
        upscaling_factor (int): number of vertical and horizontal field to convolve over.

    Returns:
        torch.tensor: low-res. output 3D tensor with updated box variables as the last 4 channels.
    """
    pixel_dim = tensor.shape[0] - 4 
    # box_dim = 4 is implicitly "hardcoded" into the structure of this function for the rectilinear case: box edges

    # Warning if upscaling is not perfect
    if (((tensor.shape[-1] % upscaling_factor) != 0) or ((tensor.shape[-2] % upscaling_factor) != 0)):
        print("ACHTUNG: Upscaling is not closed/has remainder: Mean aggregation is over fields of different sizes. Consider using a different magnification factor.")
        # exit because box upscaling does not work
        exit

    # Initialise empty target tensor to append to
    target_tensor = torch.empty(size = (0, int(tensor.shape[-2] / upscaling_factor) , int(tensor.shape[-1] / upscaling_factor)))

    # define upscaling function with torch https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html
    # We upscale the same amount in x & y direction: square window
    upscale = torch.nn.AvgPool2d(kernel_size = upscaling_factor)

    # Loop through pixel_dim and upscale each
    for i in range(0, pixel_dim):
        # need to reassign to variable name. torch.cat() is not inplace
        target_tensor = torch.cat((target_tensor, upscale(tensor[i, :, :].unsqueeze(0))), dim = 0)

    # define max_pool function with same kernel_size as upscaling
    max_pool = torch.nn.MaxPool2d(kernel_size = upscaling_factor)

    # y_min: No min pooling function: double negative
    target_tensor = torch.cat((target_tensor, (- max_pool(- tensor[pixel_dim, :, :].unsqueeze(0)))), dim = 0)
    # y_max
    target_tensor = torch.cat((target_tensor, (max_pool(tensor[(pixel_dim + 1), :, :].unsqueeze(0)))), dim = 0)
    # x_min
    target_tensor = torch.cat((target_tensor, (- max_pool(- tensor[(pixel_dim + 2), :, :].unsqueeze(0)))), dim = 0)
    # x_max
    target_tensor = torch.cat((target_tensor, (max_pool(tensor[(pixel_dim + 3), :, :].unsqueeze(0)))), dim = 0)

    return target_tensor

In [141]:
# upscale by 2 and check
lr_scene_vel_pixel_box_tensor = upscale_box_tensor(scence_vel_pixel_box_tensor, upscaling_factor = 2)

# Check if it works as expected.
# Upscaled to a 900m grid
for i in range(0, int(lr_scene_vel_pixel_box_tensor.shape[0])): 
    print(f"Channel: {i}")
    print(scence_vel_pixel_box_tensor[i, 0:2, 0:2])
    print(lr_scene_vel_pixel_box_tensor[i, 0, 0])

Channel: 0
tensor([[-21.8497, -21.8022],
        [ -5.4622,  -5.4622]], dtype=torch.float64)
tensor(-13.6441, dtype=torch.float64)
Channel: 1
tensor([[10.2547,  9.5673],
        [ 1.7139,  1.7139]], dtype=torch.float64)
tensor(5.8125, dtype=torch.float64)
Channel: 2
tensor([[ -575.,  -575.],
        [-1025., -1025.]], dtype=torch.float64)
tensor(-1025., dtype=torch.float64)
Channel: 3
tensor([[-125., -125.],
        [-575., -575.]], dtype=torch.float64)
tensor(-125., dtype=torch.float64)
Channel: 4
tensor([[-332875., -332425.],
        [-332875., -332425.]], dtype=torch.float64)
tensor(-332875., dtype=torch.float64)
Channel: 5
tensor([[-332425., -331975.],
        [-332425., -331975.]], dtype=torch.float64)
tensor(-331975., dtype=torch.float64)


In [122]:
lr_scene_vel_pixel_box_tensor[2:, :, :].shape

torch.Size([4, 25, 25])

## Task

**HR Input**: HR base_covar with box-channels [1 + 4, hr_dim, r_dim]  
- HR is the surface elevation. 
- Base covariance but at least space the domain of LR
- 500m resolution

**LR Input**: LR box-channels [4, lr_dim, lr_dim]
- It was upscaled to a 900m reoslution (from a natural 450m resolution)

Outputs:
- k_ah_al: high-low [hr_dim^2, lr_dim^2]
- k_al_al: low-low [lr_dim^2, lr_dim^2]

Example:
- hr_dim^2 = 46^2 = 2116
- lr_dim^2 = 25^2 = 625

In [180]:
scene_sur_midpoints46_tensor = torch.load("./torch_data/scene_sur_midpoints46_tensor.pt")
base_covar46 = torch.load("./torch_data/base_covar46.pt")
# box grid of surface but also of bed. Not using bed here.
scene_sur_box46_tensor = midpoint_to_box(scene_sur_midpoints46_tensor)

In [181]:
hr_dims_flat = 46**2

# create copys for all columns ([4, 2116, 1]) -> ([4, 2116, 2116]) and all rows
# flatten first for pariwise covariance representation
# all column vectors of row_box channels are the same because these correspond to rows. [BC, R, C]: box-channels, rows, columns
row_box_channels = scene_sur_box46_tensor.reshape(4, -1).unsqueeze(-1).repeat(1, 1, hr_dims_flat)
# .unsqueeze(-2) creates explicit dimension we wanna copy across (middle dimension)
column_box_channels = scene_sur_box46_tensor.reshape(4, -1).unsqueeze(-2).repeat(1, hr_dims_flat, 1)

In [182]:
# Asserting behaviour
row_box_channels[:, :, 0] == row_box_channels[:, :, 22]
column_box_channels[:, 0, :] == column_box_channels[:, 57, :]

tensor([[True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True],
        [True, True, True,  ..., True, True, True]])

In [187]:
base_covar_box = torch.cat((base_covar46.unsqueeze(0), row_box_channels, column_box_channels), dim = 0)

In [190]:
### Check that base_covar covers full area
# y_min: rows and columns contain the same so choose row. Could find min based on on position but use torch.min instead
torch.min(base_covar_box[1, :, :]) # min of y_min
torch.max(base_covar_box[2, :, :]) # max of y_max
torch.min(base_covar_box[3, :, :]) # min of x_min
torch.max(base_covar_box[4, :, :]) # max of x_max

tensor(-310250.)